# Image-Only CLIP Fine-Tuning: Tuning And Robustness Experiments

This notebook contains the self-contained partial CLIP image-encoder fine-tuning pipeline used for configuration selection and matched-seed robustness evaluation.

Completed workflow:
1. verified CSV split loading and image-path remapping on matched seeds
2. ran a one-seed parameter search over several trial configurations
3. selected the best setup using **validation macro-F1** only
4. compared the top two configurations on additional matched seeds
5. reran the selected configuration cleanly on the default seed
6. ran the final selected configuration on all 30 matched seeds

Primary output roots:
- `experiments/imageonly_clip_finetuned_pilot/`
- `experiments/imageonly_clip_finetuned_robustness/`

In [1]:
from pathlib import Path
from urllib.parse import unquote
import json
import os
import random
import time
import warnings

import clip
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

warnings.filterwarnings("ignore")

_walk = Path.cwd().resolve()
for _ in range(10):
    if (_walk / "experiments").is_dir() and (_walk / "data").is_dir():
        PROJECT_ROOT = _walk
        break
    _walk = _walk.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

BASE_DIR = PROJECT_ROOT
SPLITS_ROOT = PROJECT_ROOT / "data" / "splits"
DATASET_CSV = PROJECT_ROOT / "data" / "processed" / "caption_dataset_final_full.csv"
OUTPUT_ROOT = PROJECT_ROOT / "experiments" / "imageonly_clip_finetuned_pilot"
METRICS_DIR = OUTPUT_ROOT / "metrics"
ARTIFACTS_DIR = OUTPUT_ROOT / "artifacts"
(METRICS_DIR / "trials").mkdir(parents=True, exist_ok=True)
(METRICS_DIR / "experiments").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "trial_models").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "trial_learning_curves").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "learning_curves").mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_INIT_SEED = 42
DEFAULT_SEED = 13
DEBUG_SAMPLES = 5
NUM_WORKERS = 2
CLIP_MODEL_NAME = "ViT-B/32"

TRIAL_CONFIGS = [
    {
        "name": "trial_a_blocks2",
        "clip_model": CLIP_MODEL_NAME,
        "batch_size": 32,
        "max_epochs": 15,
        "patience": 4,
        "dropout": 0.5,
        "weight_decay": 1e-4,
        "head_lr": 5e-5,
        "clip_lr": 1e-5,
        "unfreeze_visual_blocks": 2,
    },
    {
        "name": "trial_b_blocks1",
        "clip_model": CLIP_MODEL_NAME,
        "batch_size": 32,
        "max_epochs": 15,
        "patience": 4,
        "dropout": 0.5,
        "weight_decay": 1e-4,
        "head_lr": 5e-5,
        "clip_lr": 5e-6,
        "unfreeze_visual_blocks": 1,
    },
    {
        "name": "trial_c_more_head_lr",
        "clip_model": CLIP_MODEL_NAME,
        "batch_size": 32,
        "max_epochs": 15,
        "patience": 4,
        "dropout": 0.4,
        "weight_decay": 1e-4,
        "head_lr": 1e-4,
        "clip_lr": 1e-5,
        "unfreeze_visual_blocks": 2,
    },
    {
        "name": "trial_d_blocks2_lower_clip_lr",
        "clip_model": CLIP_MODEL_NAME,
        "batch_size": 32,
        "max_epochs": 15,
        "patience": 4,
        "dropout": 0.5,
        "weight_decay": 1e-4,
        "head_lr": 1e-4,
        "clip_lr": 2e-6,
        "unfreeze_visual_blocks": 2,
    },
    {
        "name": "trial_e_blocks4_low_lr",
        "clip_model": CLIP_MODEL_NAME,
        "batch_size": 32,
        "max_epochs": 15,
        "patience": 4,
        "dropout": 0.5,
        "weight_decay": 1e-4,
        "head_lr": 1e-4,
        "clip_lr": 2e-6,
        "unfreeze_visual_blocks": 4,
    },
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLITS_ROOT:", SPLITS_ROOT)
print("DATASET_CSV:", DATASET_CSV)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DEVICE:", DEVICE)
print("Trial configs:")
for cfg in TRIAL_CONFIGS:
    print(" ", cfg)


/home/ding-zhang/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /home/ding-zhang/Dongmei/DATA255/Project
SPLITS_ROOT: /home/ding-zhang/Dongmei/DATA255/Project/data/splits
DATASET_CSV: /home/ding-zhang/Dongmei/DATA255/Project/data/processed/caption_dataset_final_full.csv
OUTPUT_ROOT: /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_pilot
DEVICE: cuda
Trial configs:
  {'name': 'trial_a_blocks2', 'clip_model': 'ViT-B/32', 'batch_size': 32, 'max_epochs': 15, 'patience': 4, 'dropout': 0.5, 'weight_decay': 0.0001, 'head_lr': 5e-05, 'clip_lr': 1e-05, 'unfreeze_visual_blocks': 2}
  {'name': 'trial_b_blocks1', 'clip_model': 'ViT-B/32', 'batch_size': 32, 'max_epochs': 15, 'patience': 4, 'dropout': 0.5, 'weight_decay': 0.0001, 'head_lr': 5e-05, 'clip_lr': 5e-06, 'unfreeze_visual_blocks': 1}
  {'name': 'trial_c_more_head_lr', 'clip_model': 'ViT-B/32', 'batch_size': 32, 'max_epochs': 15, 'patience': 4, 'dropout': 0.4, 'weight_decay': 0.0001, 'head_lr': 0.0001, 'clip_lr': 1e-05, 'unfreeze_visual_blocks': 2}
  {'name': '

## 1. Load Metadata, Split CSVs, And Verify Image Paths

This section loads shared metadata and performs initial path-resolution checks before training:
- load style metadata
- load a selected seed from `data/splits/seed_<seed>/`
- verify that `dataset/...` paths remap into `data/raw dataset/...`
- confirm that sample images exist before launching training runs

In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_style_metadata(dataset_csv):
    df = pd.read_csv(dataset_csv)
    df = df[df["status"] == "success"].copy()
    df["style"] = df["style"].str.strip()
    all_styles = sorted(df["style"].unique())
    style_to_idx = {style: idx for idx, style in enumerate(all_styles)}
    return all_styles, style_to_idx, len(all_styles)


def load_split_csvs(seed):
    split_dir = SPLITS_ROOT / f"seed_{seed}"
    train_df = pd.read_csv(split_dir / "train.csv")
    val_df = pd.read_csv(split_dir / "val.csv")
    test_df = pd.read_csv(split_dir / "test.csv")
    return split_dir, train_df, val_df, test_df


def resolve_image_path(image_path, base_dir):
    image_path = str(image_path)
    if not os.path.isabs(image_path):
        rel_path = image_path.replace("\\", "/")
        dataset_root = base_dir / "dataset"
        if rel_path.startswith("dataset/") and not dataset_root.is_dir():
            image_path = str(base_dir / "data" / "raw dataset" / rel_path[len("dataset/"):])
        else:
            image_path = str(base_dir / image_path)

    if "%" in image_path:
        parts = image_path.split("/")
        image_path = "/".join(unquote(part) if "%" in part else part for part in parts)

    return os.path.normpath(image_path)


def debug_path_resolution(df, base_dir=BASE_DIR, sample_count=DEBUG_SAMPLES):
    print("=== Path resolution sanity check ===")
    print("BASE_DIR:", base_dir)
    print("Top-level dataset/ exists:", (base_dir / "dataset").is_dir())
    print("data/raw dataset/ exists:", (base_dir / "data" / "raw dataset").is_dir())

    checked = 0
    found = 0
    for _, row in df.head(sample_count).iterrows():
        raw_path = row["image_path"]
        resolved_path = resolve_image_path(raw_path, base_dir)
        exists = os.path.isfile(resolved_path)
        found += int(exists)
        checked += 1
        print(f"  exists={exists} raw={raw_path}")
        print(f"         resolved={resolved_path}")

    print(f"Resolved files found: {found}/{checked}")
    if found != checked:
        raise RuntimeError("Some pilot image paths did not resolve correctly.")


all_styles, style_to_idx, num_classes = load_style_metadata(DATASET_CSV)
split_dir, train_df, val_df, test_df = load_split_csvs(DEFAULT_SEED)

print("Loaded seed:", DEFAULT_SEED)
print("Split dir:", split_dir)
print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Classes:", num_classes)
debug_path_resolution(train_df)


Loaded seed: 13
Split dir: /home/ding-zhang/Dongmei/DATA255/Project/data/splits/seed_13
Split sizes: 9261 1984 1985
Classes: 14
=== Path resolution sanity check ===
BASE_DIR: /home/ding-zhang/Dongmei/DATA255/Project
Top-level dataset/ exists: False
data/raw dataset/ exists: True
  exists=True raw=dataset/mode/9234303ea1b3437acd91437fb29df533.jpg
         resolved=/home/ding-zhang/Dongmei/DATA255/Project/data/raw dataset/mode/9234303ea1b3437acd91437fb29df533.jpg
  exists=True raw=dataset/conservative/deec85adc5c257cecbecd14ad2c6241999eb1fc4.jpg
         resolved=/home/ding-zhang/Dongmei/DATA255/Project/data/raw dataset/conservative/deec85adc5c257cecbecd14ad2c6241999eb1fc4.jpg
  exists=True raw=dataset/retro/a8a3be106a52c363efff4aad6bd427c0.jpg
         resolved=/home/ding-zhang/Dongmei/DATA255/Project/data/raw dataset/retro/a8a3be106a52c363efff4aad6bd427c0.jpg
  exists=True raw=dataset/conservative/m_93b41ab3faace1063459db9e7bab6137bf3d7dca1449738231.jpg
         resolved=/home/ding-zha

## 2. Dataset, Model, And Training Helpers

This section defines the reusable components for the fine-tuned CLIP image-only experiments:
- dataset loading and image-path resolution
- partial CLIP visual encoder unfreezing
- optimizer parameter groups for `clip_lr` and `head_lr`
- training, validation, and per-class metric computation
- deterministic DataLoader seeding and prediction saving
- a single-trial runner for one matched seed

In [3]:
class FashionImageOnlyDataset(Dataset):
    def __init__(self, df, style_to_idx, transform=None, base_dir=None):
        self.df = df.reset_index(drop=True)
        self.style_to_idx = style_to_idx
        self.transform = transform
        self.base_dir = Path(base_dir) if base_dir is not None else None
        self.valid_indices = []
        self.missing_examples = []

        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            resolved_path = resolve_image_path(row["image_path"], self.base_dir or PROJECT_ROOT)
            if os.path.exists(resolved_path):
                self.valid_indices.append(idx)
            elif len(self.missing_examples) < 5:
                self.missing_examples.append((str(row["image_path"]), resolved_path))

        print(f"Dataset initialized with {len(self.valid_indices)} valid samples (out of {len(self.df)})")
        if self.missing_examples:
            print("  Missing image examples:")
            for raw_path, resolved_path in self.missing_examples:
                print(f"    raw={raw_path}")
                print(f"    resolved={resolved_path}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]
        image_path = resolve_image_path(row["image_path"], self.base_dir or PROJECT_ROOT)
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        style = row["style"]
        label = self.style_to_idx[style]
        return {
            "image": image,
            "label": label,
            "style": style,
            "image_path": image_path,
        }


class ImageOnlyFashionClassifier(nn.Module):
    def __init__(self, clip_model, num_classes, dropout=0.5, visual_dim=512):
        super().__init__()
        self.clip_model = clip_model
        self.classifier = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, images):
        visual_features = self.clip_model.encode_image(images).float()
        return self.classifier(visual_features)


def configure_partial_visual_finetuning(model, unfreeze_visual_blocks):
    for param in model.clip_model.parameters():
        param.requires_grad = False

    visual = model.clip_model.visual
    trainable_names = []

    if hasattr(visual, "transformer") and hasattr(visual.transformer, "resblocks"):
        blocks = list(visual.transformer.resblocks)
        unfreeze_visual_blocks = max(0, min(unfreeze_visual_blocks, len(blocks)))
        for block_idx in range(len(blocks) - unfreeze_visual_blocks, len(blocks)):
            if block_idx < 0:
                continue
            for name, param in blocks[block_idx].named_parameters():
                param.requires_grad = True
                trainable_names.append(f"clip_model.visual.transformer.resblocks.{block_idx}.{name}")

    if hasattr(visual, "ln_post"):
        for name, param in visual.ln_post.named_parameters():
            param.requires_grad = True
            trainable_names.append(f"clip_model.visual.ln_post.{name}")

    if hasattr(visual, "proj") and isinstance(visual.proj, torch.nn.Parameter):
        visual.proj.requires_grad = True
        trainable_names.append("clip_model.visual.proj")

    for name, param in model.classifier.named_parameters():
        param.requires_grad = True
        trainable_names.append(f"classifier.{name}")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    return {
        "trainable_parameter_names": trainable_names,
        "trainable_parameter_count": int(trainable_params),
        "total_parameter_count": int(total_params),
        "unfrozen_visual_blocks": int(unfreeze_visual_blocks),
    }


def build_optimizer(model, clip_lr, head_lr, weight_decay):
    clip_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("clip_model.visual"):
            clip_params.append(param)
        else:
            head_params.append(param)

    param_groups = []
    if clip_params:
        param_groups.append({"params": clip_params, "lr": clip_lr, "name": "clip"})
    if head_params:
        param_groups.append({"params": head_params, "lr": head_lr, "name": "head"})
    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)


def train_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in tqdm(data_loader, desc="Training", leave=False):
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return total_loss / len(data_loader), 100.0 * correct / total


def validate_epoch(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Validation", leave=False):
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_predictions, average="macro", zero_division=0)
    return total_loss / len(data_loader), 100.0 * correct / total, all_predictions, all_labels, macro_f1


def evaluate_with_per_class_metrics(model, data_loader, criterion, device, all_styles):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    all_image_paths = []
    all_style_names = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating", leave=False):
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_image_paths.extend(list(batch["image_path"]))
            all_style_names.extend(list(batch["style"]))

    accuracy = 100.0 * correct / total
    avg_loss = total_loss / len(data_loader)
    macro_f1 = f1_score(all_labels, all_predictions, average="macro", zero_division=0)
    macro_precision = precision_score(all_labels, all_predictions, average="macro", zero_division=0)
    macro_recall = recall_score(all_labels, all_predictions, average="macro", zero_division=0)
    per_class_f1 = f1_score(all_labels, all_predictions, average=None, zero_division=0)
    per_class_precision = precision_score(all_labels, all_predictions, average=None, zero_division=0)
    per_class_recall = recall_score(all_labels, all_predictions, average=None, zero_division=0)

    per_class_accuracy = []
    labels_np = np.array(all_labels)
    predictions_np = np.array(all_predictions)
    for class_idx in range(len(all_styles)):
        class_mask = labels_np == class_idx
        class_acc = float(np.sum((predictions_np == class_idx) & class_mask) / np.sum(class_mask)) if np.sum(class_mask) > 0 else 0.0
        per_class_accuracy.append(class_acc)

    per_class_f1_dict = {all_styles[i]: float(per_class_f1[i]) for i in range(len(all_styles))}
    per_class_precision_dict = {all_styles[i]: float(per_class_precision[i]) for i in range(len(all_styles))}
    per_class_recall_dict = {all_styles[i]: float(per_class_recall[i]) for i in range(len(all_styles))}
    per_class_accuracy_dict = {all_styles[i]: float(per_class_accuracy[i]) for i in range(len(all_styles))}

    return (
        avg_loss,
        accuracy,
        all_predictions,
        all_labels,
        all_image_paths,
        all_style_names,
        macro_f1,
        macro_precision,
        macro_recall,
        per_class_f1_dict,
        per_class_precision_dict,
        per_class_recall_dict,
        per_class_accuracy_dict,
    )


def save_learning_curves(train_losses, val_losses, train_accs, val_accs, val_macro_f1s, clip_lrs, head_lrs, best_epoch, output_path, seed):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].plot(train_losses, label="Train Loss", color="blue", linewidth=2)
    axes[0, 0].plot(val_losses, label="Val Loss", color="red", linewidth=2)
    axes[0, 0].axvline(x=best_epoch - 1, color="green", linestyle="--", alpha=0.7)
    axes[0, 0].set_title("Training and Validation Loss")
    axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(train_accs, label="Train Acc", color="blue", linewidth=2)
    axes[0, 1].plot(val_accs, label="Val Acc", color="red", linewidth=2)
    axes[0, 1].axvline(x=best_epoch - 1, color="green", linestyle="--", alpha=0.7)
    axes[0, 1].set_title("Training and Validation Accuracy")
    axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(val_macro_f1s, label="Val Macro F1", color="green", linewidth=2)
    axes[1, 0].axvline(x=best_epoch - 1, color="red", linestyle="--", alpha=0.7)
    axes[1, 0].set_title("Validation Macro F1")
    axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(head_lrs, label="Head LR", color="purple", linewidth=2)
    if clip_lrs:
        axes[1, 1].plot(clip_lrs, label="CLIP LR", color="orange", linewidth=2)
    axes[1, 1].set_title("Learning Rates")
    axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f"Image-only CLIP fine-tune pilot: seed {seed}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def run_finetune_trial(config, seed=DEFAULT_SEED, save_as_seed_result=False):
    set_seed(MODEL_INIT_SEED)
    split_dir, train_df, val_df, test_df = load_split_csvs(seed)

    clip_model, clip_preprocess = clip.load(config.get("clip_model", CLIP_MODEL_NAME), device=DEVICE, jit=False)
    clip_model = clip_model.float()
    model = ImageOnlyFashionClassifier(
        clip_model=clip_model,
        num_classes=num_classes,
        dropout=config["dropout"],
        visual_dim=int(getattr(clip_model.visual, "output_dim", 512)),
    ).to(DEVICE)

    trainable_info = configure_partial_visual_finetuning(model, config["unfreeze_visual_blocks"])
    print("\n=== Trainable parameter summary ===")
    print("Trial:", config["name"])
    print("Unfrozen visual blocks:", trainable_info["unfrozen_visual_blocks"])
    print("Trainable parameters:", trainable_info["trainable_parameter_count"])
    for name in trainable_info["trainable_parameter_names"][:20]:
        print("  ", name)
    if len(trainable_info["trainable_parameter_names"]) > 20:
        print("  ...")

    train_dataset = FashionImageOnlyDataset(train_df, style_to_idx, clip_preprocess, base_dir=BASE_DIR)
    val_dataset = FashionImageOnlyDataset(val_df, style_to_idx, clip_preprocess, base_dir=BASE_DIR)
    test_dataset = FashionImageOnlyDataset(test_df, style_to_idx, clip_preprocess, base_dir=BASE_DIR)

    worker_seed_base = MODEL_INIT_SEED + seed

    def seed_worker(worker_id):
        worker_seed = worker_seed_base + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    g = torch.Generator()
    g.manual_seed(worker_seed_base)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        worker_init_fn=seed_worker,
        generator=g,
    )
    val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=NUM_WORKERS)

    train_valid_df = train_dataset.df.iloc[train_dataset.valid_indices]
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array(list(style_to_idx.values())),
        y=train_valid_df["style"].map(style_to_idx).values,
    )
    class_weights = torch.FloatTensor(class_weights).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = build_optimizer(model, config["clip_lr"], config["head_lr"], config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])

    train_losses, val_losses, train_accs, val_accs, val_macro_f1s = [], [], [], [], []
    clip_lrs, head_lrs = [], []
    best_val_macro_f1 = -1.0
    min_val_loss = float("inf")
    best_epoch = 0
    best_train_loss = None
    best_train_acc = None
    best_val_loss_at_epoch = None
    best_val_acc_at_epoch = None
    patience_counter = 0
    early_stopped = False
    start_time = time.time()

    if save_as_seed_result:
        model_path = ARTIFACTS_DIR / "models" / f"seed_{seed}_best_model.pth"
        curve_path = ARTIFACTS_DIR / "learning_curves" / f"seed_{seed}_learning_curves.png"
        result_path = METRICS_DIR / "experiments" / f"seed_{seed}_results.json"
    else:
        model_path = ARTIFACTS_DIR / "trial_models" / f"{config['name']}_seed_{seed}.pth"
        curve_path = ARTIFACTS_DIR / "trial_learning_curves" / f"{config['name']}_seed_{seed}.png"
        result_path = METRICS_DIR / "trials" / f"{config['name']}_seed_{seed}.json"

    print(f"\n{'=' * 70}")
    print(f"Running trial {config['name']} on seed {seed}")
    print(f"Loaded CSVs from {split_dir}")
    print(f"Train/Val/Test: {len(train_df)} / {len(val_df)} / {len(test_df)}")
    print(f"{'=' * 70}")

    for epoch in range(config["max_epochs"]):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, _, _, val_macro_f1 = validate_epoch(model, val_loader, criterion, DEVICE)

        current_clip_lr = 0.0
        current_head_lr = 0.0
        for group in optimizer.param_groups:
            if group.get("name") == "clip":
                current_clip_lr = float(group["lr"])
            else:
                current_head_lr = float(group["lr"])

        clip_lrs.append(current_clip_lr)
        head_lrs.append(current_head_lr)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        val_macro_f1s.append(val_macro_f1)

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch + 1
            best_train_loss = train_loss
            best_train_acc = train_acc
            best_val_loss_at_epoch = val_loss
            best_val_acc_at_epoch = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1

        if val_loss < min_val_loss:
            min_val_loss = val_loss

        print(
            f"Epoch {epoch + 1}/{config['max_epochs']}: "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
            f"val_macro_f1={val_macro_f1:.4f} clip_lr={current_clip_lr:.2e} head_lr={current_head_lr:.2e}"
        )

        scheduler.step()
        if patience_counter >= config["patience"]:
            early_stopped = True
            print(f"Early stopping at epoch {epoch + 1}")
            break

    total_time = time.time() - start_time
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))

    (
        val_loss,
        val_acc,
        val_predictions,
        val_labels,
        val_image_paths,
        val_styles,
        val_macro_f1,
        val_macro_precision,
        val_macro_recall,
        val_per_class_f1,
        val_per_class_precision,
        val_per_class_recall,
        val_per_class_accuracy,
    ) = evaluate_with_per_class_metrics(model, val_loader, criterion, DEVICE, all_styles)

    (
        test_loss,
        test_acc,
        test_predictions,
        test_labels,
        test_image_paths,
        test_styles,
        test_macro_f1,
        test_macro_precision,
        test_macro_recall,
        test_per_class_f1,
        test_per_class_precision,
        test_per_class_recall,
        test_per_class_accuracy,
    ) = evaluate_with_per_class_metrics(model, test_loader, criterion, DEVICE, all_styles)

    recomputed_val_loss_best_checkpoint = float(val_loss)
    recomputed_val_macro_f1_best_checkpoint = float(val_macro_f1)
    val_loss_at_best_epoch = float(best_val_loss_at_epoch) if best_val_loss_at_epoch is not None else recomputed_val_loss_best_checkpoint
    val_accuracy_at_best_epoch = float(best_val_acc_at_epoch) if best_val_acc_at_epoch is not None else float(val_acc)
    final_val_loss = float(val_losses[-1]) if val_losses else recomputed_val_loss_best_checkpoint
    overfitting_detected = final_val_loss > val_loss_at_best_epoch * 1.05

    train_val_gap = float(best_train_loss - val_loss_at_best_epoch) if best_train_loss is not None else 0.0
    save_learning_curves(train_losses, val_losses, train_accs, val_accs, val_macro_f1s, clip_lrs, head_lrs, best_epoch, curve_path, seed)

    results = {
        "experiment_id": f"seed_{seed}",
        "seed_value": seed,
        "seed_index": 1,
        "trial_name": config["name"],
        "model_type": "image_only_clip_finetuned_partial",
        "configuration": {
            **config,
            "model_init_seed": MODEL_INIT_SEED,
            "data_split_seed": seed,
            "split_source": "csv",
        },
        "training_info": {
            "total_epochs": len(train_losses),
            "best_epoch": best_epoch,
            "early_stopped": early_stopped,
            "total_time_minutes": float(total_time / 60.0),
            "trainable_parameter_count": trainable_info["trainable_parameter_count"],
            "train_loss_at_best_epoch": float(best_train_loss) if best_train_loss is not None else None,
            "train_accuracy_at_best_epoch": float(best_train_acc) if best_train_acc is not None else None,
        },
        "validation_metrics": {
            "best_val_macro_f1": float(best_val_macro_f1),
            "best_val_accuracy": float(val_accuracy_at_best_epoch),
            "best_val_macro_precision": float(val_macro_precision),
            "best_val_macro_recall": float(val_macro_recall),
            "val_loss_at_best_epoch": float(val_loss_at_best_epoch),
            "val_accuracy_at_best_epoch": float(val_accuracy_at_best_epoch),
            "recomputed_val_macro_f1_best_checkpoint": float(recomputed_val_macro_f1_best_checkpoint),
            "recomputed_val_loss_best_checkpoint": float(recomputed_val_loss_best_checkpoint),
            "min_val_loss": float(min_val_loss),
            "predictions": [int(x) for x in val_predictions],
            "labels": [int(x) for x in val_labels],
            "image_paths": [str(x) for x in val_image_paths],
            "styles": [str(x) for x in val_styles],
            "per_class_metrics": {
                "f1": val_per_class_f1,
                "precision": val_per_class_precision,
                "recall": val_per_class_recall,
                "accuracy": val_per_class_accuracy,
            },
        },
        "test_metrics": {
            "test_macro_f1": float(test_macro_f1),
            "test_accuracy": float(test_acc),
            "test_macro_precision": float(test_macro_precision),
            "test_macro_recall": float(test_macro_recall),
            "test_loss": float(test_loss),
            "predictions": [int(x) for x in test_predictions],
            "labels": [int(x) for x in test_labels],
            "image_paths": [str(x) for x in test_image_paths],
            "styles": [str(x) for x in test_styles],
            "per_class_metrics": {
                "f1": test_per_class_f1,
                "precision": test_per_class_precision,
                "recall": test_per_class_recall,
                "accuracy": test_per_class_accuracy,
            },
        },
        "overfitting_analysis": {
            "overfitting_detected": overfitting_detected,
            "val_loss_at_best_epoch": float(val_loss_at_best_epoch),
            "min_val_loss": float(min_val_loss),
            "final_val_loss": float(final_val_loss),
            "train_val_gap": float(train_val_gap),
        },
        "training_curves": {
            "train_losses": [float(x) for x in train_losses],
            "val_losses": [float(x) for x in val_losses],
            "train_accs": [float(x) for x in train_accs],
            "val_accs": [float(x) for x in val_accs],
            "val_macro_f1s": [float(x) for x in val_macro_f1s],
            "clip_learning_rates": [float(x) for x in clip_lrs],
            "head_learning_rates": [float(x) for x in head_lrs],
        },
        "data_split_info": {
            "train_size": len(train_df),
            "val_size": len(val_df),
            "test_size": len(test_df),
        },
        "trainable_parameters": trainable_info,
    }

    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"Saved result JSON to {result_path}")
    print(f"Best val macro F1: {results['validation_metrics']['best_val_macro_f1']:.4f}")
    print(f"Test macro F1: {results['test_metrics']['test_macro_f1']:.4f}")
    return results


## 3. One-Seed Parameter Search

This section runs a small set of partial CLIP fine-tuning trials on one matched seed (`DEFAULT_SEED`) and ranks them by validation performance.

Configuration selection is **validation-only**: `BEST_CONFIG` is chosen using validation macro-F1 and validation loss. Test metrics are logged for reference but are not used for model selection.

In [4]:
SEARCH_RESULTS = []
for trial_cfg in TRIAL_CONFIGS:
    result = run_finetune_trial(trial_cfg, seed=DEFAULT_SEED, save_as_seed_result=False)
    SEARCH_RESULTS.append(
        {
            "trial_name": trial_cfg["name"],
            "best_val_macro_f1": result["validation_metrics"]["best_val_macro_f1"],
            "val_loss_at_best_epoch": result["validation_metrics"]["val_loss_at_best_epoch"],
            "min_val_loss": result["validation_metrics"]["min_val_loss"],
            "test_macro_f1": result["test_metrics"]["test_macro_f1"],
            "epochs": result["training_info"]["total_epochs"],
            "clip_lr": trial_cfg["clip_lr"],
            "head_lr": trial_cfg["head_lr"],
            "unfreeze_visual_blocks": trial_cfg["unfreeze_visual_blocks"],
            "dropout": trial_cfg["dropout"],
            "weight_decay": trial_cfg["weight_decay"],
        }
    )

df_search = pd.DataFrame(SEARCH_RESULTS).sort_values(
    ["best_val_macro_f1", "val_loss_at_best_epoch", "epochs"],
    ascending=[False, True, True],
).reset_index(drop=True)

display_cols = [
    "trial_name",
    "best_val_macro_f1",
    "val_loss_at_best_epoch",
    "min_val_loss",
    "epochs",
    "clip_lr",
    "head_lr",
    "unfreeze_visual_blocks",
    "dropout",
    "weight_decay",
]
display(df_search[display_cols])

BEST_CONFIG = next(cfg for cfg in TRIAL_CONFIGS if cfg["name"] == df_search.iloc[0]["trial_name"]).copy()
TOP2_CONFIG_NAMES = df_search.head(2)["trial_name"].tolist()
print("Best config from validation-only search:")
print(BEST_CONFIG)
print("Top 2 configs:", TOP2_CONFIG_NAMES)



=== Trainable parameter summary ===
Trial: trial_a_blocks2
Unfrozen visual blocks: 2
Trainable parameters: 14736526
   clip_model.visual.transformer.resblocks.10.attn.in_proj_weight
   clip_model.visual.transformer.resblocks.10.attn.in_proj_bias
   clip_model.visual.transformer.resblocks.10.attn.out_proj.weight
   clip_model.visual.transformer.resblocks.10.attn.out_proj.bias
   clip_model.visual.transformer.resblocks.10.ln_1.weight
   clip_model.visual.transformer.resblocks.10.ln_1.bias
   clip_model.visual.transformer.resblocks.10.mlp.c_fc.weight
   clip_model.visual.transformer.resblocks.10.mlp.c_fc.bias
   clip_model.visual.transformer.resblocks.10.mlp.c_proj.weight
   clip_model.visual.transformer.resblocks.10.mlp.c_proj.bias
   clip_model.visual.transformer.resblocks.10.ln_2.weight
   clip_model.visual.transformer.resblocks.10.ln_2.bias
   clip_model.visual.transformer.resblocks.11.attn.in_proj_weight
   clip_model.visual.transformer.resblocks.11.attn.in_proj_bias
   clip_model.v

Epoch 1/15: train_loss=2.1838 val_loss=1.2499 val_macro_f1=0.6864 clip_lr=1.00e-05 head_lr=5.00e-05


Epoch 2/15: train_loss=1.0637 val_loss=0.6032 val_macro_f1=0.8112 clip_lr=9.89e-06 head_lr=4.95e-05


Epoch 3/15: train_loss=0.6598 val_loss=0.4844 val_macro_f1=0.8394 clip_lr=9.57e-06 head_lr=4.78e-05


Epoch 4/15: train_loss=0.4554 val_loss=0.4727 val_macro_f1=0.8517 clip_lr=9.05e-06 head_lr=4.52e-05


...

Epoch 13/15: train_loss=0.1023 val_loss=0.5755 val_macro_f1=0.8546 clip_lr=1.91e-07 head_lr=9.55e-06


Epoch 14/15: train_loss=0.0977 val_loss=0.5776 val_macro_f1=0.8562 clip_lr=8.65e-08 head_lr=4.32e-06
Early stopping at epoch 14


Saved result JSON to /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_pilot/metrics/trials/trial_e_blocks4_low_lr_seed_13.json
Best val macro F1: 0.8607
Test macro F1: 0.8518


,trial_name,best_val_macro_f1,val_loss_at_best_epoch,min_val_loss,epochs,clip_lr,head_lr,unfreeze_visual_blocks,dropout,weight_decay
0,trial_a_blocks2,0.862665,0.551011,0.472718,11,0.000010,0.00005,2,0.5,0.0001
1,trial_c_more_head_lr,0.861886,0.666093,0.448189,14,0.000010,0.00010,2,0.4,0.0001
2,trial_e_blocks4_low_lr,0.860674,0.536244,0.457864,14,0.000002,0.00010,4,0.5,0.0001
3,trial_d_blocks2_lower_clip_lr,0.856537,0.448380,0.448153,14,0.000002,0.00010,2,0.5,0.0001
4,trial_b_blocks1,0.848731,0.437365,0.437365,15,0.000005,0.00005,1,0.5,0.0001


Best config from validation-only search:
{'name': 'trial_a_blocks2', 'clip_model': 'ViT-B/32', 'batch_size': 32, 'max_epochs': 15, 'patience': 4, 'dropout': 0.5, 'weight_decay': 0.0001, 'head_lr': 5e-05, 'clip_lr': 1e-05, 'unfreeze_visual_blocks': 2}
Top 2 configs: ['trial_a_blocks2', 'trial_c_more_head_lr']


## 3.5 Shortlist Comparison On Additional Seeds

This section compares the top two trial configurations on four matched seeds (seed 13 plus three additional seeds). Existing seed-13 trial results are reused when available. The final `BEST_CONFIG` is selected from the aggregated validation macro-F1 summary.

In [ ]:
SHORTLIST_CONFIG_NAMES = ["trial_a_blocks2", "trial_c_more_head_lr"]
SHORTLIST_NEW_SEEDS = [14, 17, 53]
SHORTLIST_ALL_SEEDS = [DEFAULT_SEED] + SHORTLIST_NEW_SEEDS  # seed 13 + 3 more

trial_cfg_lookup = {cfg["name"]: cfg for cfg in TRIAL_CONFIGS}
SHORTLIST_CONFIGS = [trial_cfg_lookup[name].copy() for name in SHORTLIST_CONFIG_NAMES]

def load_existing_trial_result(trial_name, seed):
    result_path = METRICS_DIR / "trials" / f"{trial_name}_seed_{seed}.json"
    if result_path.exists():
        with open(result_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

SHORTLIST_RUN_RESULTS = []

for trial_cfg in SHORTLIST_CONFIGS:
    for seed in SHORTLIST_ALL_SEEDS:
        existing_result = load_existing_trial_result(trial_cfg["name"], seed)

        if existing_result is not None:
            result = existing_result
            print(f"Reusing existing result: {trial_cfg['name']} seed {seed}")
        else:
            result = run_finetune_trial(trial_cfg, seed=seed, save_as_seed_result=False)

        SHORTLIST_RUN_RESULTS.append(
            {
                "trial_name": trial_cfg["name"],
                "seed": seed,
                "best_val_macro_f1": result["validation_metrics"]["best_val_macro_f1"],
                "val_loss_at_best_epoch": result["validation_metrics"]["val_loss_at_best_epoch"],
                "min_val_loss": result["validation_metrics"]["min_val_loss"],
                "test_macro_f1": result["test_metrics"]["test_macro_f1"],
                "epochs": result["training_info"]["total_epochs"],
                "best_epoch": result["training_info"]["best_epoch"],
                "early_stopped": result["training_info"]["early_stopped"],
                "total_time_minutes": result["training_info"]["total_time_minutes"],
                "clip_lr": trial_cfg["clip_lr"],
                "head_lr": trial_cfg["head_lr"],
                "unfreeze_visual_blocks": trial_cfg["unfreeze_visual_blocks"],
                "dropout": trial_cfg["dropout"],
                "weight_decay": trial_cfg["weight_decay"],
            }
        )

df_shortlist_runs = (
    pd.DataFrame(SHORTLIST_RUN_RESULTS)
    .sort_values(["trial_name", "seed"])
    .reset_index(drop=True)
)

print("\nPer-seed shortlist results:")
display(
    df_shortlist_runs[
        [
            "trial_name",
            "seed",
            "best_val_macro_f1",
            "val_loss_at_best_epoch",
            "test_macro_f1",
            "epochs",
            "best_epoch",
            "early_stopped",
            "total_time_minutes",
            "clip_lr",
            "head_lr",
            "unfreeze_visual_blocks",
        ]
    ]
)

df_shortlist_summary = (
    df_shortlist_runs
    .groupby("trial_name", as_index=False)
    .agg(
        mean_val_macro_f1=("best_val_macro_f1", "mean"),
        std_val_macro_f1=("best_val_macro_f1", "std"),
        mean_val_loss_at_best_epoch=("val_loss_at_best_epoch", "mean"),
        mean_test_macro_f1=("test_macro_f1", "mean"),
        std_test_macro_f1=("test_macro_f1", "std"),
        mean_epochs=("epochs", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_time_minutes=("total_time_minutes", "mean"),
        clip_lr=("clip_lr", "first"),
        head_lr=("head_lr", "first"),
        unfreeze_visual_blocks=("unfreeze_visual_blocks", "first"),
        dropout=("dropout", "first"),
        weight_decay=("weight_decay", "first"),
    )
)

df_shortlist_summary["std_val_macro_f1"] = df_shortlist_summary["std_val_macro_f1"].fillna(0.0)
df_shortlist_summary["std_test_macro_f1"] = df_shortlist_summary["std_test_macro_f1"].fillna(0.0)

df_shortlist_summary = df_shortlist_summary.sort_values(
    ["mean_val_macro_f1", "std_val_macro_f1", "mean_val_loss_at_best_epoch"],
    ascending=[False, True, True],
).reset_index(drop=True)

print("\nAggregated shortlist summary (selection is validation-only):")
display(
    df_shortlist_summary[
        [
            "trial_name",
            "mean_val_macro_f1",
            "std_val_macro_f1",
            "mean_val_loss_at_best_epoch",
            "mean_test_macro_f1",
            "std_test_macro_f1",
            "mean_epochs",
            "mean_best_epoch",
            "mean_time_minutes",
            "clip_lr",
            "head_lr",
            "unfreeze_visual_blocks",
        ]
    ]
)

BEST_CONFIG = next(
    cfg for cfg in TRIAL_CONFIGS if cfg["name"] == df_shortlist_summary.iloc[0]["trial_name"]
).copy()

print("\nSelected BEST_CONFIG after shortlist:")
print(BEST_CONFIG)
print(f"Section 4 will now rerun this config cleanly on DEFAULT_SEED={DEFAULT_SEED}.")

Reusing existing result: trial_a_blocks2 seed 13

=== Trainable parameter summary ===
Trial: trial_a_blocks2
Unfrozen visual blocks: 2
Trainable parameters: 14736526
   clip_model.visual.transformer.resblocks.10.attn.in_proj_weight
   clip_model.visual.transformer.resblocks.10.attn.in_proj_bias
   clip_model.visual.transformer.resblocks.10.attn.out_proj.weight
   clip_model.visual.transformer.resblocks.10.attn.out_proj.bias
   clip_model.visual.transformer.resblocks.10.ln_1.weight
   clip_model.visual.transformer.resblocks.10.ln_1.bias
   clip_model.visual.transformer.resblocks.10.mlp.c_fc.weight
   clip_model.visual.transformer.resblocks.10.mlp.c_fc.bias
   clip_model.visual.transformer.resblocks.10.mlp.c_proj.weight
   clip_model.visual.transformer.resblocks.10.mlp.c_proj.bias
   clip_model.visual.transformer.resblocks.10.ln_2.weight
   clip_model.visual.transformer.resblocks.10.ln_2.bias
   clip_model.visual.transformer.resblocks.11.attn.in_proj_weight
   clip_model.visual.transform

Epoch 1/15: train_loss=2.1921 val_loss=1.2509 val_macro_f1=0.7098 clip_lr=1.00e-05 head_lr=5.00e-05


Epoch 2/15: train_loss=1.0485 val_loss=0.6049 val_macro_f1=0.8045 clip_lr=9.89e-06 head_lr=4.95e-05


Epoch 3/15: train_loss=0.6561 val_loss=0.4930 val_macro_f1=0.8271 clip_lr=9.57e-06 head_lr=4.78e-05


Epoch 4/15: train_loss=0.4477 val_loss=0.4511 val_macro_f1=0.8518 clip_lr=9.05e-06 head_lr=4.52e-05


...

Saved result JSON to /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_pilot/metrics/trials/trial_c_more_head_lr_seed_53.json
Best val macro F1: 0.8423
Test macro F1: 0.8511

Per-seed shortlist results:


,trial_name,seed,best_val_macro_f1,val_loss_at_best_epoch,test_macro_f1,epochs,best_epoch,early_stopped,total_time_minutes,clip_lr,head_lr,unfreeze_visual_blocks
0,trial_a_blocks2,13,0.862665,0.551011,0.853308,11,7,True,1.835367,0.00001,0.00005,2
1,trial_a_blocks2,14,0.865193,0.617986,0.848306,15,14,False,2.536229,0.00001,0.00005,2
2,trial_a_blocks2,17,0.854974,0.624486,0.840151,13,9,True,2.202599,0.00001,0.00005,2
3,trial_a_blocks2,53,0.843979,0.707818,0.856102,14,10,True,2.394369,0.00001,0.00005,2
4,trial_c_more_head_lr,13,0.861886,0.666093,0.856343,14,10,True,2.343153,0.00001,0.00010,2
5,trial_c_more_head_lr,14,0.867044,0.632039,0.854144,15,13,False,2.553466,0.00001,0.00010,2
6,trial_c_more_head_lr,17,0.853495,0.568908,0.844461,10,6,True,1.703112,0.00001,0.00010,2
7,trial_c_more_head_lr,53,0.842333,0.802206,0.851123,15,15,False,2.567267,0.00001,0.00010,2



Aggregated shortlist summary (selection is validation-only):


,trial_name,mean_val_macro_f1,std_val_macro_f1,mean_val_loss_at_best_epoch,mean_test_macro_f1,std_test_macro_f1,mean_epochs,mean_best_epoch,mean_time_minutes,clip_lr,head_lr,unfreeze_visual_blocks
0,trial_a_blocks2,0.856703,0.009531,0.625325,0.849467,0.006998,13.25,10.0,2.242141,0.00001,0.00005,2
1,trial_c_more_head_lr,0.856190,0.010794,0.667312,0.851518,0.005168,13.50,11.0,2.291749,0.00001,0.00010,2



Selected BEST_CONFIG after shortlist:
{'name': 'trial_a_blocks2', 'clip_model': 'ViT-B/32', 'batch_size': 32, 'max_epochs': 15, 'patience': 4, 'dropout': 0.5, 'weight_decay': 0.0001, 'head_lr': 5e-05, 'clip_lr': 1e-05, 'unfreeze_visual_blocks': 2}
Section 4 will now rerun this config cleanly on DEFAULT_SEED=13.


## 4. Clean Rerun Of The Selected Configuration

This section reruns the selected partial CLIP configuration cleanly on the default seed after the search and shortlist stages, producing the final one-seed result before the full 30-seed robustness run.

In [6]:
if "BEST_CONFIG" not in globals():
    BEST_CONFIG = TRIAL_CONFIGS[0].copy()

BEST_CONFIG = BEST_CONFIG.copy()
BEST_CONFIG["name"] = "selected_config"
BEST_CONFIG["max_epochs"] = 20
BEST_CONFIG["patience"] = 5

print("Final config to rerun:")
print(BEST_CONFIG)

final_result = run_finetune_trial(BEST_CONFIG, seed=DEFAULT_SEED, save_as_seed_result=True)

result_path = METRICS_DIR / "experiments" / f"seed_{DEFAULT_SEED}_results.json"
print("\nFinal result JSON:", result_path)
print("Best val macro F1:", final_result["validation_metrics"]["best_val_macro_f1"])
print("Test macro F1:", final_result["test_metrics"]["test_macro_f1"])
print("Trainable params:", final_result["training_info"]["trainable_parameter_count"])


Final config to rerun:
{'name': 'selected_config', 'clip_model': 'ViT-B/32', 'batch_size': 32, 'max_epochs': 20, 'patience': 5, 'dropout': 0.5, 'weight_decay': 0.0001, 'head_lr': 5e-05, 'clip_lr': 1e-05, 'unfreeze_visual_blocks': 2}

=== Trainable parameter summary ===
Trial: selected_config
Unfrozen visual blocks: 2
Trainable parameters: 14736526
   clip_model.visual.transformer.resblocks.10.attn.in_proj_weight
   clip_model.visual.transformer.resblocks.10.attn.in_proj_bias
   clip_model.visual.transformer.resblocks.10.attn.out_proj.weight
   clip_model.visual.transformer.resblocks.10.attn.out_proj.bias
   clip_model.visual.transformer.resblocks.10.ln_1.weight
   clip_model.visual.transformer.resblocks.10.ln_1.bias
   clip_model.visual.transformer.resblocks.10.mlp.c_fc.weight
   clip_model.visual.transformer.resblocks.10.mlp.c_fc.bias
   clip_model.visual.transformer.resblocks.10.mlp.c_proj.weight
   clip_model.visual.transformer.resblocks.10.mlp.c_proj.bias
   clip_model.visual.trans

Epoch 1/20: train_loss=2.1838 val_loss=1.2499 val_macro_f1=0.6864 clip_lr=1.00e-05 head_lr=5.00e-05


Epoch 2/20: train_loss=1.0623 val_loss=0.6007 val_macro_f1=0.8119 clip_lr=9.94e-06 head_lr=4.97e-05


Epoch 3/20: train_loss=0.6574 val_loss=0.4845 val_macro_f1=0.8388 clip_lr=9.76e-06 head_lr=4.88e-05


Epoch 4/20: train_loss=0.4523 val_loss=0.4727 val_macro_f1=0.8520 clip_lr=9.46e-06 head_lr=4.73e-05


...

Epoch 9/20: train_loss=0.0831 val_loss=0.6532 val_macro_f1=0.8534 clip_lr=6.55e-06 head_lr=3.27e-05


Epoch 10/20: train_loss=0.0631 val_loss=0.6799 val_macro_f1=0.8588 clip_lr=5.78e-06 head_lr=2.89e-05


Epoch 11/20: train_loss=0.0463 val_loss=0.7391 val_macro_f1=0.8537 clip_lr=5.00e-06 head_lr=2.50e-05


Epoch 12/20: train_loss=0.0333 val_loss=0.7707 val_macro_f1=0.8537 clip_lr=4.22e-06 head_lr=2.11e-05
Early stopping at epoch 12


Saved result JSON to /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_pilot/metrics/experiments/seed_13_results.json
Best val macro F1: 0.8642
Test macro F1: 0.8565

Final result JSON: /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_pilot/metrics/experiments/seed_13_results.json
Best val macro F1: 0.8641769630481492
Test macro F1: 0.8564586192028225
Trainable params: 14736526


## 5. Full 30-Seed Robustness Run

This section runs the final selected partial CLIP configuration across all 30 matched seeds. It saves per-seed JSON results, aggregated summary tables, statistical summaries, learning curves, and comparison plots under `experiments/imageonly_clip_finetuned_robustness/`.

In [8]:
import re

try:
    from scipy import stats as scipy_stats
except Exception:
    scipy_stats = None

if "BEST_CONFIG" not in globals():
    raise RuntimeError("Run section 4 first so BEST_CONFIG is defined.")

if "all_styles" not in globals() or "style_to_idx" not in globals() or "num_classes" not in globals():
    all_styles, style_to_idx, num_classes = load_style_metadata(DATASET_CSV)

ROBUSTNESS_OUTPUT_ROOT = PROJECT_ROOT / "experiments" / "imageonly_clip_finetuned_robustness"
ROBUSTNESS_METRICS_DIR = ROBUSTNESS_OUTPUT_ROOT / "metrics"

# Use root-level folders so the layout is closer to older robustness notebook outputs.
ROBUSTNESS_ARTIFACTS_DIR = ROBUSTNESS_OUTPUT_ROOT
(ROBUSTNESS_METRICS_DIR / "trials").mkdir(parents=True, exist_ok=True)
(ROBUSTNESS_METRICS_DIR / "experiments").mkdir(parents=True, exist_ok=True)
(ROBUSTNESS_ARTIFACTS_DIR / "models").mkdir(parents=True, exist_ok=True)
(ROBUSTNESS_ARTIFACTS_DIR / "learning_curves").mkdir(parents=True, exist_ok=True)
(ROBUSTNESS_ARTIFACTS_DIR / "comparison_plots").mkdir(parents=True, exist_ok=True)

SEEDS_FILE = PROJECT_ROOT / "experiments" / "phase3_robustness" / "metrics" / "seeds_list.txt"

def load_robustness_seeds():
    if SEEDS_FILE.exists():
        text = SEEDS_FILE.read_text(encoding="utf-8")
        seeds = [int(x) for x in re.findall(r"Seed\s+(\d+)", text)]
        if seeds:
            return seeds

    seeds = []
    for p in sorted(SPLITS_ROOT.glob("seed_*")):
        if p.is_dir() and (p / "train.csv").exists() and (p / "val.csv").exists() and (p / "test.csv").exists():
            try:
                seeds.append(int(p.name.split("_")[1]))
            except Exception:
                pass
    return seeds

def calculate_stats(values, name):
    arr = np.array(values, dtype=float)
    mean_val = float(np.mean(arr))
    std_val = float(np.std(arr))
    min_val = float(np.min(arr))
    max_val = float(np.max(arr))
    median_val = float(np.median(arr))
    q25 = float(np.percentile(arr, 25))
    q75 = float(np.percentile(arr, 75))
    cv = float((std_val / mean_val * 100.0) if mean_val != 0 else 0.0)

    if len(arr) > 1 and scipy_stats is not None:
        ci = scipy_stats.t.interval(0.95, len(arr) - 1, loc=mean_val, scale=scipy_stats.sem(arr))
        ci_lower = float(ci[0])
        ci_upper = float(ci[1])
    else:
        ci_lower = mean_val
        ci_upper = mean_val

    return {
        "metric": name,
        "mean": mean_val,
        "std": std_val,
        "min": min_val,
        "max": max_val,
        "median": median_val,
        "q25": q25,
        "q75": q75,
        "cv_percent": cv,
        "ci_95_lower": ci_lower,
        "ci_95_upper": ci_upper,
        "n": int(len(arr)),
    }

def safe_to_excel(df, path):
    try:
        df.to_excel(path, index=False)
        print(f"Saved Excel: {path}")
    except Exception as exc:
        print(f"Could not save Excel file {path}: {exc}")

def fmt_mean_std(mean_val, std_val, digits=4):
    return f"{mean_val:.{digits}f} +/- {std_val:.{digits}f}"

def fmt_ci(stat_obj, digits=4):
    return f"[{stat_obj['ci_95_lower']:.{digits}f}, {stat_obj['ci_95_upper']:.{digits}f}]"

def write_robustness_seeds_list(metrics_dir, seeds):
    path = metrics_dir / "seeds_list.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write("Fine-tuned image-only robustness seeds\n")
        f.write("=" * 50 + "\n")
        f.write(f"Total seeds: {len(seeds)}\n\n")
        for idx, seed in enumerate(seeds, 1):
            f.write(f"{idx:2d}. Seed {seed}\n")

def build_per_class_tables(all_results, all_styles):
    summary_rows = []
    per_class_stats = {}
    detailed_tables = {
        "f1": [],
        "precision": [],
        "recall": [],
        "accuracy": [],
    }

    for style in all_styles:
        style_metrics = {
            "f1": [],
            "precision": [],
            "recall": [],
            "accuracy": [],
        }

        for result in all_results:
            test_pc = result["test_metrics"].get("per_class_metrics", {})
            if not test_pc:
                continue
            for metric_name in style_metrics:
                metric_dict = test_pc.get(metric_name, {})
                if style in metric_dict:
                    style_metrics[metric_name].append(float(metric_dict[style]))

        if not style_metrics["f1"]:
            continue

        style_stat_block = {
            metric_name: calculate_stats(values, f"{style} - Test {metric_name.title()}")
            for metric_name, values in style_metrics.items()
        }
        per_class_stats[style] = style_stat_block

        summary_rows.append(
            {
                "Style": style,
                "Test_F1_Mean": style_stat_block["f1"]["mean"],
                "Test_F1_Std": style_stat_block["f1"]["std"],
                "Test_Precision_Mean": style_stat_block["precision"]["mean"],
                "Test_Precision_Std": style_stat_block["precision"]["std"],
                "Test_Recall_Mean": style_stat_block["recall"]["mean"],
                "Test_Recall_Std": style_stat_block["recall"]["std"],
                "Test_Accuracy_Mean": style_stat_block["accuracy"]["mean"],
                "Test_Accuracy_Std": style_stat_block["accuracy"]["std"],
            }
        )

        for metric_name in ["f1", "precision", "recall", "accuracy"]:
            stat_obj = style_stat_block[metric_name]
            detailed_tables[metric_name].append(
                {
                    "Style": style,
                    "Mean": stat_obj["mean"],
                    "Std": stat_obj["std"],
                    "Min": stat_obj["min"],
                    "Max": stat_obj["max"],
                    "95% CI Lower": stat_obj["ci_95_lower"],
                    "95% CI Upper": stat_obj["ci_95_upper"],
                    "N": stat_obj["n"],
                }
            )

    df_per_class_summary = pd.DataFrame(summary_rows).sort_values("Style").reset_index(drop=True)
    detailed_tables = {
        metric_name: pd.DataFrame(rows).sort_values("Style").reset_index(drop=True)
        for metric_name, rows in detailed_tables.items()
    }
    return df_per_class_summary, per_class_stats, detailed_tables




In [9]:
def save_robustness_plots(all_results, df_summary, df_per_class_summary):
    if len(all_results) == 0:
        return

    plot_dir = ROBUSTNESS_ARTIFACTS_DIR / "comparison_plots"
    plot_dir.mkdir(parents=True, exist_ok=True)

    results_sorted = sorted(all_results, key=lambda x: x["seed_value"])
    seeds = [r["seed_value"] for r in results_sorted]
    test_f1s = [r["test_metrics"]["test_macro_f1"] for r in results_sorted]
    test_accs = [r["test_metrics"]["test_accuracy"] for r in results_sorted]
    val_f1s = [r["validation_metrics"]["best_val_macro_f1"] for r in results_sorted]
    train_times = [r["training_info"]["total_time_minutes"] for r in results_sorted]

    # 1. Robustness summary plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(seeds, test_f1s, marker="o", linewidth=2, color="tab:blue")
    axes[0, 0].axhline(np.mean(test_f1s), color="red", linestyle="--", label=f"Mean={np.mean(test_f1s):.4f}")
    axes[0, 0].set_title("Test Macro F1 Across Seeds")
    axes[0, 0].set_xlabel("Seed")
    axes[0, 0].set_ylabel("Macro F1")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(seeds, test_accs, marker="o", linewidth=2, color="tab:green")
    axes[0, 1].axhline(np.mean(test_accs), color="red", linestyle="--", label=f"Mean={np.mean(test_accs):.2f}")
    axes[0, 1].set_title("Test Accuracy Across Seeds")
    axes[0, 1].set_xlabel("Seed")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].boxplot([val_f1s, test_f1s], labels=["Best Val Macro F1", "Test Macro F1"])
    axes[1, 0].set_title("Validation vs Test Macro F1")
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(seeds, train_times, marker="o", linewidth=2, color="tab:purple")
    axes[1, 1].axhline(np.mean(train_times), color="red", linestyle="--", label=f"Mean={np.mean(train_times):.2f} min")
    axes[1, 1].set_title("Training Time Across Seeds")
    axes[1, 1].set_xlabel("Seed")
    axes[1, 1].set_ylabel("Minutes")
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()

    plt.tight_layout()
    robustness_plot_path = plot_dir / "robustness_analysis.png"
    plt.savefig(robustness_plot_path, dpi=300, bbox_inches="tight")
    plt.close()

    # 2. Learning curve overlay (up to first 5 completed seeds)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    overlay_results = results_sorted[: min(5, len(results_sorted))]

    for result in overlay_results:
        seed = result["seed_value"]
        curves = result.get("training_curves", {})
        val_curve = curves.get("val_macro_f1s", [])
        val_loss_curve = curves.get("val_losses", [])
        if val_curve:
            axes[0].plot(range(1, len(val_curve) + 1), val_curve, linewidth=2, label=f"Seed {seed}")
        if val_loss_curve:
            axes[1].plot(range(1, len(val_loss_curve) + 1), val_loss_curve, linewidth=2, label=f"Seed {seed}")

    axes[0].set_title("Validation Macro F1 Overlay")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Macro F1")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].set_title("Validation Loss Overlay")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    overlay_path = plot_dir / "learning_curves_overlay.png"
    plt.savefig(overlay_path, dpi=300, bbox_inches="tight")
    plt.close()

    # 3. Per-class F1 summary plot
    if len(df_per_class_summary) > 0:
        df_plot = df_per_class_summary.sort_values("Test_F1_Mean", ascending=True)

        plt.figure(figsize=(10, max(6, len(df_plot) * 0.45)))
        plt.barh(
            df_plot["Style"],
            df_plot["Test_F1_Mean"],
            xerr=df_plot["Test_F1_Std"],
            color="steelblue",
            alpha=0.85,
            ecolor="black",
            capsize=4,
        )
        plt.xlabel("Test F1 Mean")
        plt.title("Per-Class Test F1 Summary")
        plt.grid(True, axis="x", alpha=0.3)
        plt.tight_layout()

        per_class_plot_path = ROBUSTNESS_OUTPUT_ROOT / "per_class_analysis_summary.png"
        plt.savefig(per_class_plot_path, dpi=300, bbox_inches="tight")
        plt.close()

    

def save_robustness_outputs(all_results, failed_seeds, seeds, metrics_dir, all_styles):
    summary = {
        "total_seeds": len(seeds),
        "successful_experiments": len(all_results),
        "failed_experiments": len(failed_seeds),
        "failed_seeds": failed_seeds,
        "completed_seeds": [result["seed_value"] for result in all_results],
        "experiment_root": str(ROBUSTNESS_OUTPUT_ROOT),
    }
    with open(metrics_dir / "experiments_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    if not all_results:
        return

    summary_rows = []
    for result in all_results:
        validation_metrics = result["validation_metrics"]
        test_metrics = result["test_metrics"]
        overfit_metrics = result["overfitting_analysis"]
        training_info = result["training_info"]

        summary_rows.append(
            {
                "Seed": result["seed_value"],
                "Best_Epoch": training_info["best_epoch"],
                "Early_Stopped": training_info["early_stopped"],
                "Best_Val_Macro_F1": validation_metrics["best_val_macro_f1"],
                "Best_Val_Accuracy": validation_metrics["best_val_accuracy"],
                "Val_Loss_At_Best_Epoch": validation_metrics.get(
                    "val_loss_at_best_epoch",
                    validation_metrics.get("best_val_loss")
                ),
                "Min_Val_Loss": validation_metrics.get(
                    "min_val_loss",
                    validation_metrics.get("best_val_loss")
                ),
                "Test_Macro_F1": test_metrics["test_macro_f1"],
                "Test_Accuracy": test_metrics["test_accuracy"],
                "Test_Macro_Precision": test_metrics["test_macro_precision"],
                "Test_Macro_Recall": test_metrics["test_macro_recall"],
                "Overfitting": overfit_metrics["overfitting_detected"],
                "Training_Time_Min": training_info["total_time_minutes"],
                "Trainable_Params": training_info["trainable_parameter_count"],
            }
        )

    df_summary = pd.DataFrame(summary_rows).sort_values("Seed").reset_index(drop=True)
    df_summary.to_csv(metrics_dir / "summary_table.csv", index=False)
    safe_to_excel(df_summary, metrics_dir / "summary_table.xlsx")

    overall_stats = [
        calculate_stats(df_summary["Test_Macro_F1"].tolist(), "Test Macro F1"),
        calculate_stats(df_summary["Test_Accuracy"].tolist(), "Test Accuracy"),
        calculate_stats(df_summary["Test_Macro_Precision"].tolist(), "Test Macro Precision"),
        calculate_stats(df_summary["Test_Macro_Recall"].tolist(), "Test Macro Recall"),
        calculate_stats(df_summary["Best_Val_Macro_F1"].tolist(), "Best Val Macro F1"),
        calculate_stats(df_summary["Best_Val_Accuracy"].tolist(), "Best Val Accuracy"),
        calculate_stats(df_summary["Val_Loss_At_Best_Epoch"].tolist(), "Val Loss At Best Epoch"),
        calculate_stats(df_summary["Min_Val_Loss"].tolist(), "Min Val Loss"),
        calculate_stats(df_summary["Training_Time_Min"].tolist(), "Training Time (min)"),
    ]

    with open(metrics_dir / "overall_metrics_statistics.json", "w", encoding="utf-8") as f:
        json.dump({"overall_metrics": overall_stats}, f, indent=2)
    pd.DataFrame(overall_stats).to_csv(metrics_dir / "overall_metrics_summary.csv", index=False)

    overfitting_count = int(df_summary["Overfitting"].sum())
    statistical_analysis = {
        "summary_statistics": overall_stats,
        "overfitting_count": overfitting_count,
        "overfitting_percentage": float(100.0 * overfitting_count / len(df_summary)),
        "total_experiments": int(len(df_summary)),
    }
    with open(metrics_dir / "statistical_analysis.json", "w", encoding="utf-8") as f:
        json.dump(statistical_analysis, f, indent=2)

    # Report-friendly overall table
    overall_report_rows = []
    for stat_obj in overall_stats:
        digits = 4 if "Accuracy" not in stat_obj["metric"] and "Time" not in stat_obj["metric"] else 4
        overall_report_rows.append(
            {
                "Metric": stat_obj["metric"],
                "Mean +/- Std": fmt_mean_std(stat_obj["mean"], stat_obj["std"], digits=digits),
                "95% CI": fmt_ci(stat_obj, digits=digits),
                "Min": round(stat_obj["min"], digits),
                "Max": round(stat_obj["max"], digits),
                "CV %": round(stat_obj["cv_percent"], 2),
            }
        )
    pd.DataFrame(overall_report_rows).to_csv(metrics_dir / "comprehensive_report_overall.csv", index=False)

    # Per-class outputs
    df_per_class_summary, per_class_stats, detailed_tables = build_per_class_tables(all_results, all_styles)
    with open(metrics_dir / "per_class_metrics_statistics.json", "w", encoding="utf-8") as f:
        json.dump(per_class_stats, f, indent=2)

    if len(df_per_class_summary) > 0:
        df_per_class_summary.to_csv(metrics_dir / "per_class_metrics_summary.csv", index=False)
        safe_to_excel(df_per_class_summary, metrics_dir / "per_class_metrics_summary.xlsx")

        detailed_tables["f1"].to_csv(metrics_dir / "per_class_metrics_f1_summary.csv", index=False)
        detailed_tables["precision"].to_csv(metrics_dir / "per_class_metrics_precision_summary.csv", index=False)
        detailed_tables["recall"].to_csv(metrics_dir / "per_class_metrics_recall_summary.csv", index=False)
        detailed_tables["accuracy"].to_csv(metrics_dir / "per_class_metrics_accuracy_summary.csv", index=False)

        # Report-friendly per-class table
        per_class_report_rows = []
        for style in sorted(per_class_stats.keys()):
            f1_stat = per_class_stats[style]["f1"]
            precision_stat = per_class_stats[style]["precision"]
            recall_stat = per_class_stats[style]["recall"]
            per_class_report_rows.append(
                {
                    "Style": style,
                    "F1-Score": fmt_mean_std(f1_stat["mean"], f1_stat["std"], digits=4),
                    "Precision": fmt_mean_std(precision_stat["mean"], precision_stat["std"], digits=4),
                    "Recall": fmt_mean_std(recall_stat["mean"], recall_stat["std"], digits=4),
                    "95% CI (F1)": fmt_ci(f1_stat, digits=4),
                }
            )
        pd.DataFrame(per_class_report_rows).to_csv(metrics_dir / "comprehensive_report_per_class.csv", index=False)

    save_robustness_plots(all_results, df_summary, df_per_class_summary)

In [ ]:


ROBUSTNESS_CONFIG = BEST_CONFIG.copy()
ROBUSTNESS_CONFIG["name"] = "selected_config_30_seed"

# Prevent accidental mixing of results from different configs.
selected_config_path = ROBUSTNESS_METRICS_DIR / "selected_config.json"
config_keys_to_compare = [
    "clip_model",
    "batch_size",
    "max_epochs",
    "patience",
    "dropout",
    "weight_decay",
    "head_lr",
    "clip_lr",
    "unfreeze_visual_blocks",
]
if selected_config_path.exists():
    previous_config = json.loads(selected_config_path.read_text(encoding="utf-8"))
    old_core = {k: previous_config.get(k) for k in config_keys_to_compare}
    new_core = {k: ROBUSTNESS_CONFIG.get(k) for k in config_keys_to_compare}
    if old_core != new_core:
        raise RuntimeError(
            "The robustness output folder already contains results from a different config. "
            "Use a new output folder or remove the old robustness results first."
        )

with open(selected_config_path, "w", encoding="utf-8") as f:
    json.dump(ROBUSTNESS_CONFIG, f, indent=2)

ROBUSTNESS_SEEDS = load_robustness_seeds()
write_robustness_seeds_list(ROBUSTNESS_METRICS_DIR, ROBUSTNESS_SEEDS)

print("Running final 30-seed robustness with config:")
print(ROBUSTNESS_CONFIG)
print(f"Total seeds found: {len(ROBUSTNESS_SEEDS)}")
print(f"Output root: {ROBUSTNESS_OUTPUT_ROOT}")

ORIG_OUTPUT_ROOT = OUTPUT_ROOT
ORIG_METRICS_DIR = METRICS_DIR
ORIG_ARTIFACTS_DIR = ARTIFACTS_DIR

all_results = []
failed_seeds = []

try:
    OUTPUT_ROOT = ROBUSTNESS_OUTPUT_ROOT
    METRICS_DIR = ROBUSTNESS_METRICS_DIR
    ARTIFACTS_DIR = ROBUSTNESS_ARTIFACTS_DIR

    for seed_idx, seed in enumerate(ROBUSTNESS_SEEDS, start=1):
        result_path = METRICS_DIR / "experiments" / f"seed_{seed}_results.json"

        try:
            if result_path.exists():
                print(f"[{seed_idx}/{len(ROBUSTNESS_SEEDS)}] Reusing saved result for seed {seed}")
                with open(result_path, "r", encoding="utf-8") as f:
                    result = json.load(f)
            else:
                print(f"[{seed_idx}/{len(ROBUSTNESS_SEEDS)}] Running seed {seed}")
                result = run_finetune_trial(ROBUSTNESS_CONFIG, seed=seed, save_as_seed_result=True)

            all_results.append(result)

        except Exception as exc:
            print(f"[{seed_idx}/{len(ROBUSTNESS_SEEDS)}] Seed {seed} failed: {exc}")
            failed_seeds.append({"seed": seed, "error": str(exc)})

        # Refresh summary outputs after each seed so the run is safely resumable.
        save_robustness_outputs(
            all_results=all_results,
            failed_seeds=failed_seeds,
            seeds=ROBUSTNESS_SEEDS,
            metrics_dir=ROBUSTNESS_METRICS_DIR,
            all_styles=all_styles,
        )

finally:
    OUTPUT_ROOT = ORIG_OUTPUT_ROOT
    METRICS_DIR = ORIG_METRICS_DIR
    ARTIFACTS_DIR = ORIG_ARTIFACTS_DIR



Running final 30-seed robustness with config:
{'name': 'selected_config_30_seed', 'clip_model': 'ViT-B/32', 'batch_size': 32, 'max_epochs': 20, 'patience': 5, 'dropout': 0.5, 'weight_decay': 0.0001, 'head_lr': 5e-05, 'clip_lr': 1e-05, 'unfreeze_visual_blocks': 2}
Total seeds found: 30
Output root: /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_robustness
[1/30] Running seed 13

=== Trainable parameter summary ===
Trial: selected_config_30_seed
Unfrozen visual blocks: 2
Trainable parameters: 14736526
   clip_model.visual.transformer.resblocks.10.attn.in_proj_weight
   clip_model.visual.transformer.resblocks.10.attn.in_proj_bias
   clip_model.visual.transformer.resblocks.10.attn.out_proj.weight
   clip_model.visual.transformer.resblocks.10.attn.out_proj.bias
   clip_model.visual.transformer.resblocks.10.ln_1.weight
   clip_model.visual.transformer.resblocks.10.ln_1.bias
   clip_model.visual.transformer.resblocks.10.mlp.c_fc.weight
   clip_model.visual.trans

Epoch 1/20: train_loss=2.1838 val_loss=1.2499 val_macro_f1=0.6864 clip_lr=1.00e-05 head_lr=5.00e-05


Epoch 2/20: train_loss=1.0623 val_loss=0.6007 val_macro_f1=0.8119 clip_lr=9.94e-06 head_lr=4.97e-05


Epoch 3/20: train_loss=0.6574 val_loss=0.4845 val_macro_f1=0.8388 clip_lr=9.76e-06 head_lr=4.88e-05


Epoch 4/20: train_loss=0.4523 val_loss=0.4727 val_macro_f1=0.8520 clip_lr=9.46e-06 head_lr=4.73e-05


...

Epoch 9/20: train_loss=0.0834 val_loss=0.6362 val_macro_f1=0.8481 clip_lr=6.55e-06 head_lr=3.27e-05


Epoch 10/20: train_loss=0.0654 val_loss=0.6819 val_macro_f1=0.8483 clip_lr=5.78e-06 head_lr=2.89e-05
Early stopping at epoch 10


Saved result JSON to /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_robustness/metrics/experiments/seed_457_results.json
Best val macro F1: 0.8573
Test macro F1: 0.8569
Could not save Excel file /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_robustness/metrics/summary_table.xlsx: No module named 'openpyxl'
Could not save Excel file /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_robustness/metrics/per_class_metrics_summary.xlsx: No module named 'openpyxl'

Robustness run complete.
Successful seeds: 30 / 30
Failed seeds: 0
Results saved under: /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_robustness

Per-seed summary:


,Seed,Best_Epoch,Early_Stopped,Best_Val_Macro_F1,Best_Val_Accuracy,Val_Loss_At_Best_Epoch,Min_Val_Loss,Test_Macro_F1,Test_Accuracy,Test_Macro_Precision,Test_Macro_Recall,Overfitting,Training_Time_Min,Trainable_Params
0,13,7,True,0.864177,86.434695,0.557778,0.472665,0.856459,85.598787,0.856898,0.858515,True,2.026171,14736526
1,14,12,True,0.868878,86.774356,0.646560,0.454653,0.844758,84.460141,0.846473,0.845918,True,2.873862,14736526
2,16,13,True,0.846987,84.615385,0.814394,0.525787,0.839231,83.854692,0.837304,0.842233,True,3.019219,14736526
3,17,6,True,0.852015,85.288170,0.563871,0.485463,0.842217,84.335523,0.842775,0.845592,True,1.863181,14736526
4,45,10,True,0.868400,86.740891,0.628770,0.450880,0.851487,84.883721,0.855104,0.849054,True,2.527485,14736526
5,48,12,True,0.860972,85.966683,0.666906,0.469464,0.860307,85.959596,0.861302,0.860272,True,2.864100,14736526
6,53,6,True,0.840287,83.897022,0.570595,0.524352,0.858126,85.685382,0.862317,0.856295,True,1.864216,14736526
7,58,10,True,0.851648,85.050505,0.647782,0.482702,0.862055,86.147624,0.861908,0.863061,True,2.533940,14736526
8,72,8,True,0.849684,84.906613,0.618076,0.493872,0.842042,84.367121,0.842720,0.843586,True,2.203088,14736526
9,102,16,False,0.863030,86.161616,0.750892,0.458429,0.867332,86.686838,0.869834,0.865773,False,3.390536,14736526



High-level robustness summary:
{
  "summary_statistics": [
    {
      "metric": "Test Macro F1",
      "mean": 0.850517229173512,
      "std": 0.007420450267628044,
      "min": 0.8392313864779696,
      "max": 0.8673322786940278,
      "median": 0.8493367043695952,
      "q25": 0.8443644703638196,
      "q75": 0.8553446168414114,
      "cv_percent": 0.8724632509606947,
      "ci_95_lower": 0.8476990192600071,
      "ci_95_upper": 0.8533354390870169,
      "n": 30
    },
    {
      "metric": "Test Accuracy",
      "mean": 84.98352365668964,
      "std": 0.7254833788240446,
      "min": 83.85469223007064,
      "max": 86.68683812405446,
      "median": 84.83316481294236,
      "q25": 84.39037571120417,
      "q75": 85.56088933804952,
      "cv_percent": 0.8536753332973112,
      "ci_95_lower": 84.70799260715152,
      "ci_95_upper": 85.25905470622776,
      "n": 30
    },
    {
      "metric": "Test Macro Precision",
      "mean": 0.8523073421510515,
      "std": 0.008151895519083569

In [11]:
print("\nRobustness run complete.")
print(f"Successful seeds: {len(all_results)} / {len(ROBUSTNESS_SEEDS)}")
print(f"Failed seeds: {len(failed_seeds)}")
print(f"Results saved under: {ROBUSTNESS_OUTPUT_ROOT}")

summary_csv = ROBUSTNESS_METRICS_DIR / "summary_table.csv"
overall_json = ROBUSTNESS_METRICS_DIR / "statistical_analysis.json"
experiments_summary_json = ROBUSTNESS_METRICS_DIR / "experiments_summary.json"

if summary_csv.exists():
    print("\nPer-seed summary:")
    display(pd.read_csv(summary_csv).sort_values("Seed"))

if overall_json.exists():
    print("\nHigh-level robustness summary:")
    with open(overall_json, "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2))

print("\nKey files:")
print("  ", ROBUSTNESS_METRICS_DIR / "experiments")
print("  ", ROBUSTNESS_METRICS_DIR / "summary_table.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "summary_table.xlsx")
print("  ", ROBUSTNESS_METRICS_DIR / "overall_metrics_statistics.json")
print("  ", ROBUSTNESS_METRICS_DIR / "overall_metrics_summary.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "per_class_metrics_statistics.json")
print("  ", ROBUSTNESS_METRICS_DIR / "per_class_metrics_summary.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "per_class_metrics_summary.xlsx")
print("  ", ROBUSTNESS_METRICS_DIR / "per_class_metrics_f1_summary.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "per_class_metrics_precision_summary.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "per_class_metrics_recall_summary.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "per_class_metrics_accuracy_summary.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "comprehensive_report_overall.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "comprehensive_report_per_class.csv")
print("  ", ROBUSTNESS_METRICS_DIR / "statistical_analysis.json")
print("  ", experiments_summary_json)
print("  ", ROBUSTNESS_OUTPUT_ROOT / "comparison_plots" / "robustness_analysis.png")
print("  ", ROBUSTNESS_OUTPUT_ROOT / "comparison_plots" / "learning_curves_overlay.png")
print("  ", ROBUSTNESS_OUTPUT_ROOT / "per_class_analysis_summary.png")


Robustness run complete.
Successful seeds: 30 / 30
Failed seeds: 0
Results saved under: /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_robustness

Per-seed summary:


,Seed,Best_Epoch,Early_Stopped,Best_Val_Macro_F1,Best_Val_Accuracy,Val_Loss_At_Best_Epoch,Min_Val_Loss,Test_Macro_F1,Test_Accuracy,Test_Macro_Precision,Test_Macro_Recall,Overfitting,Training_Time_Min,Trainable_Params
0,13,7,True,0.864177,86.434695,0.557778,0.472665,0.856459,85.598787,0.856898,0.858515,True,2.026171,14736526
1,14,12,True,0.868878,86.774356,0.646560,0.454653,0.844758,84.460141,0.846473,0.845918,True,2.873862,14736526
2,16,13,True,0.846987,84.615385,0.814394,0.525787,0.839231,83.854692,0.837304,0.842233,True,3.019219,14736526
3,17,6,True,0.852015,85.288170,0.563871,0.485463,0.842217,84.335523,0.842775,0.845592,True,1.863181,14736526
4,45,10,True,0.868400,86.740891,0.628770,0.450880,0.851487,84.883721,0.855104,0.849054,True,2.527485,14736526
5,48,12,True,0.860972,85.966683,0.666906,0.469464,0.860307,85.959596,0.861302,0.860272,True,2.864100,14736526
6,53,6,True,0.840287,83.897022,0.570595,0.524352,0.858126,85.685382,0.862317,0.856295,True,1.864216,14736526
7,58,10,True,0.851648,85.050505,0.647782,0.482702,0.862055,86.147624,0.861908,0.863061,True,2.533940,14736526
8,72,8,True,0.849684,84.906613,0.618076,0.493872,0.842042,84.367121,0.842720,0.843586,True,2.203088,14736526
9,102,16,False,0.863030,86.161616,0.750892,0.458429,0.867332,86.686838,0.869834,0.865773,False,3.390536,14736526



High-level robustness summary:
{
  "summary_statistics": [
    {
      "metric": "Test Macro F1",
      "mean": 0.850517229173512,
      "std": 0.007420450267628044,
      "min": 0.8392313864779696,
      "max": 0.8673322786940278,
      "median": 0.8493367043695952,
      "q25": 0.8443644703638196,
      "q75": 0.8553446168414114,
      "cv_percent": 0.8724632509606947,
      "ci_95_lower": 0.8476990192600071,
      "ci_95_upper": 0.8533354390870169,
      "n": 30
    },
    {
      "metric": "Test Accuracy",
      "mean": 84.98352365668964,
      "std": 0.7254833788240446,
      "min": 83.85469223007064,
      "max": 86.68683812405446,
      "median": 84.83316481294236,
      "q25": 84.39037571120417,
      "q75": 85.56088933804952,
      "cv_percent": 0.8536753332973112,
      "ci_95_lower": 84.70799260715152,
      "ci_95_upper": 85.25905470622776,
      "n": 30
    },
    {
      "metric": "Test Macro Precision",
      "mean": 0.8523073421510515,
      "std": 0.008151895519083569